# MarginGraph: LLM rerank go/no-go (E5/E6)

Runs on a free Kaggle T4 GPU (Colab also works). It rebuilds the CPU pipeline, so the LLM sees exactly the same top-20 LightGBM candidates, then scores each candidate with Qwen2.5 and compares the result against LightGBM on the same baskets.

**GO** only if the LLM, or an LLM + LightGBM blend, beats LightGBM on Hit@5 with a paired 95% CI above zero.

No API keys or tokens are needed.

In [ ]:
# 1) Check the GPU, then install (about 2 min). Needs Internet ON in the Kaggle settings panel.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.chdir('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
!rm -rf MarginGraph && git clone -q -b claude/project-thread-mkbaip https://github.com/Sanskar1724/MarginGraph.git
%cd MarginGraph
!pip install -q -e . transformers accelerate

In [ ]:
# 2) Find the dataset.
#    Kaggle: add your uploaded dataset (Add Input -> Your Datasets); any online_retail*.xlsx or .csv under /kaggle/input is found.
#    Colab: a file picker opens instead.
import glob, os, shutil
os.makedirs('data/raw', exist_ok=True)
found = sorted(glob.glob('/kaggle/input/**/online_retail*.xlsx', recursive=True)) + \
        sorted(glob.glob('/kaggle/input/**/online_retail*.csv', recursive=True))
if found:
    src = found[0]
elif os.path.isdir('/kaggle/input'):
    raise FileNotFoundError('No online_retail*.xlsx/.csv under /kaggle/input. Click "Add Input" and add your dataset.')
else:
    from google.colab import files
    src = next(iter(files.upload()))
dst = 'data/raw/online_retail_II' + os.path.splitext(src)[1]
shutil.copy(src, dst)
!sed -i "s#raw_path: .*#raw_path: {dst}#" configs/default.yaml
print('using', src, os.path.getsize(dst) // 1_000_000, 'MB')

In [ ]:
# 3) CPU pipeline (GPU sits idle here, that's expected): clean, baselines, LightGBM, Stage 2 (about 10-15 min on Colab)
!OPENBLAS_NUM_THREADS=1 margingraph run 2>&1 | grep -v Warning
import pandas as pd
pd.read_csv('results/e5_rerank.csv').round(4)

In [ ]:
# 4) LLM scoring. N_EVENTS=1000 takes about 5-10 min on a T4 (20 prompts per basket).
import torch
assert torch.cuda.is_available(), 'No GPU attached: Session options -> Accelerator -> GPU T4 x2, then Run All again'
print('LLM runs on', torch.cuda.get_device_name(0))
import pandas as pd
from margingraph.llm import score_slates, compare, verdict
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # then try 'Qwen/Qwen2.5-3B-Instruct'
N_EVENTS = 1000
slates = pd.read_parquet('artifacts/test_slates.parquet')
catalog = pd.read_parquet('artifacts/catalog.parquet')
ids = slates.event_id.drop_duplicates().sample(min(N_EVENTS, slates.event_id.nunique()), random_state=0)
sub = slates[slates.event_id.isin(ids)].copy()
sub['llm'] = score_slates(sub, catalog, MODEL, batch_size=32)
sub.to_parquet(f"llm_scores_{MODEL.split('/')[-1]}.parquet")
table = compare(sub)
display(table.round(4))
print(verdict(table))

## 5) Send the result back

Copy the printed table and the GO/NO-GO line into the project thread. The scores file `llm_scores_*.parquet` is in the Output panel (`/kaggle/working/MarginGraph/`).